In [1]:
import pandas as pd
import re

# 1. Membaca dataset
# Menggunakan encoding latin-1 karena sering ada karakter khusus di teks Twitter
print("Membaca file data...")
df = pd.read_csv('data.csv', encoding='latin-1')
df_alay = pd.read_csv('new_kamusalay.csv', encoding='latin-1', header=None, names=['alay', 'formal'])

# 2. Membuat dictionary dari kamus alay agar pencariannya lebih cepat
kamus_alay_dict = dict(zip(df_alay['alay'], df_alay['formal']))

# 3. Membuat fungsi pembersihan teks (Text Cleaning)
def bersihkan_teks(text):
    # a. Ubah semua huruf menjadi huruf kecil (Case Folding)
    text = str(text).lower()
    
    # b. Hapus kata 'rt', 'user', dan tautan/URL (karena ini data Twitter)
    text = re.sub(r'rt\s+|user\s+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # c. Hapus tanda baca, angka, dan karakter khusus (hanya sisakan huruf)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # d. Hapus spasi yang berlebihan
    text = re.sub(r'\s+', ' ', text).strip()
    
    # e. Normalisasi kata alay menjadi kata baku
    kata_kata = text.split()
    kata_baku = [kamus_alay_dict.get(kata, kata) for kata in kata_kata]
    text = ' '.join(kata_baku)
    
    return text

# 4. Menerapkan fungsi pembersihan ke kolom 'Tweet'
print("Mulai membersihkan teks (ini mungkin butuh waktu beberapa detik)...")
df['Tweet_Clean'] = df['Tweet'].apply(bersihkan_teks)

# 5. Memilih kolom yang penting dan menyimpannya ke file baru
# Kita ambil kolom teks yang sudah bersih dan target utamanya (HS dan Abusive)
df_bersih = df[['Tweet_Clean', 'HS', 'Abusive']]

# Simpan hasilnya ke file CSV baru agar tidak merusak data asli
df_bersih.to_csv('data_clean.csv', index=False)
print("Selesai! Data yang sudah bersih berhasil disimpan sebagai 'data_clean.csv'")

# Tampilkan 5 data teratas untuk melihat hasilnya
df_bersih.head()

Membaca file data...
Mulai membersihkan teks (ini mungkin butuh waktu beberapa detik)...
Selesai! Data yang sudah bersih berhasil disimpan sebagai 'data_clean.csv'


,Tweet_Clean,HS,Abusive
0,di saat semua cowok berusaha melacak perhatian...,1,1
1,pengguna siapa yang telat memberi tau kamu eda...,0,1
2,kadang aku berpikir kenapa aku tetap percaya p...,0,0
3,aku itu aku dan ku tau matamu sipit tapi dilih...,0,0
4,kaum cebong kafir sudah kelihatan dongoknya da...,1,1


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Baca data yang sudah bersih tadi
df_bersih = pd.read_csv('data_clean.csv')

# Pastikan tidak ada data yang kosong (NaN) setelah proses pembersihan
df_bersih = df_bersih.dropna(subset=['Tweet_Clean'])

# 2. Pisahkan Fitur (X) dan Target/Label (y)
# Kita gunakan kolom 'HS' (Hate Speech) sebagai target untuk mendeteksi cyberbullying
X = df_bersih['Tweet_Clean']
y = df_bersih['HS']

# 3. Bagi data menjadi Data Latih (Training) dan Data Uji (Testing)
# 80% data digunakan untuk mengajari model, 20% digunakan untuk mengetes kepintarannya
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Ekstraksi Fitur dengan TF-IDF (Mengubah teks menjadi bobot angka)
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 5. Latih Model Machine Learning (Menggunakan Naive Bayes)
print("Sedang melatih model...")
model_nb = MultinomialNB()
model_nb.fit(X_train_tfidf, y_train)

# 6. Evaluasi Model (Menguji akurasi model menggunakan 20% data uji)
prediksi = model_nb.predict(X_test_tfidf)
akurasi = accuracy_score(y_test, prediksi)

print("Pelatihan selesai!\n")
print(f"Akurasi Model: {akurasi * 100:.2f}%\n")
print("Laporan Detail (Classification Report):")
print(classification_report(y_test, prediksi))

Sedang melatih model...
Pelatihan selesai!

Akurasi Model: 83.75%

Laporan Detail (Classification Report):
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1514
           1       0.87      0.72      0.79      1120

    accuracy                           0.84      2634
   macro avg       0.85      0.82      0.83      2634
weighted avg       0.84      0.84      0.83      2634



In [3]:
# Buat beberapa kalimat contoh (kamu bisa menggantinya dengan kalimatmu sendiri)
kalimat_tes = [
    "wah, desain sistem navigasi cerdas yang kamu buat keren banget!",
    "",
    "muka kamu seperti jelek",
    "kamu seperti tai."
]

# 1. Bersihkan kalimat tes menggunakan fungsi bersihkan_teks yang sudah dibuat di awal
kalimat_bersih = [bersihkan_teks(teks) for teks in kalimat_tes]

# 2. Ubah teks menjadi angka menggunakan TF-IDF yang sudah dilatih (gunakan .transform, bukan .fit_transform)
kalimat_tfidf = tfidf.transform(kalimat_bersih)

# 3. Minta model melakukan prediksi
prediksi_baru = model_nb.predict(kalimat_tfidf)

# 4. Tampilkan hasil prediksinya
print("=== HASIL PREDIKSI MODEL ===\n")
for teks, hasil in zip(kalimat_tes, prediksi_baru):
    label = "🔴 CYBERBULLYING" if hasil == 1 else "🟢 AMAN (Bukan Bullying)"
    print(f"Komentar : '{teks}'")
    print(f"Deteksi  : {label}\n")

=== HASIL PREDIKSI MODEL ===

Komentar : 'wah, desain sistem navigasi cerdas yang kamu buat keren banget!'
Deteksi  : 🟢 AMAN (Bukan Bullying)

Komentar : ''
Deteksi  : 🟢 AMAN (Bukan Bullying)

Komentar : 'muka kamu seperti jelek'
Deteksi  : 🔴 CYBERBULLYING

Komentar : 'kamu seperti tai.'
Deteksi  : 🔴 CYBERBULLYING



In [5]:
from sklearn.svm import SVC

# 1. PERBAIKAN TF-IDF: 
# Kita pakai bi-gram (1,2), TAPI kita abaikan kata/frasa yang muncul kurang dari 5 kali di dataset (min_df=5)
# Kita juga batasi maksimal hanya mengambil 10.000 kata/frasa yang paling penting (max_features=10000)
tfidf_optimized = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_features=10000)

X_train_opt = tfidf_optimized.fit_transform(X_train)
X_test_opt = tfidf_optimized.transform(X_test)

# 2. Latih ulang Naive Bayes dengan TF-IDF yang sudah dioptimasi
model_nb_opt = MultinomialNB()
model_nb_opt.fit(X_train_opt, y_train)
prediksi_nb_opt = model_nb_opt.predict(X_test_opt)
akurasi_nb_opt = accuracy_score(y_test, prediksi_nb_opt)

# 3. Latih menggunakan algoritma baru: SVM (Support Vector Machine)
# SVM agak lebih lama proses pelatihannya (bisa sekitar 10-30 detik), jadi mohon tunggu sebentar ya
print("Sedang melatih model SVM... (Tunggu sebentar)")
model_svm = SVC(kernel='linear', random_state=42)
model_svm.fit(X_train_opt, y_train)
prediksi_svm = model_svm.predict(X_test_opt)
akurasi_svm = accuracy_score(y_test, prediksi_svm)

# 4. Tampilkan Perbandingan Hasil
print("=== HASIL EKSPERIMEN ===")
print(f"Akurasi Naive Bayes (Optimasi) : {akurasi_nb_opt * 100:.2f}%")
print(f"Akurasi SVM (Linear Kernel)    : {akurasi_svm * 100:.2f}%")

Sedang melatih model SVM... (Tunggu sebentar)
=== HASIL EKSPERIMEN ===
Akurasi Naive Bayes (Optimasi) : 84.17%
Akurasi SVM (Linear Kernel)    : 84.81%


In [6]:
import joblib

# 1. Menyimpan Model SVM
joblib.dump(model_svm, 'model_svm_cyberbullying.joblib')

# 2. Menyimpan fitur pembobot TF-IDF
joblib.dump(tfidf_optimized, 'tfidf_vectorizer.joblib')

print("Hore! Model dan Vectorizer berhasil disimpan ke komputermu.")

Hore! Model dan Vectorizer berhasil disimpan ke komputermu.
